# Joint OPS + CROP-Seq Heatmap

Split-triangle heatmap of pairwise Pearson correlations between perturbation embeddings:
- **Upper triangle**: OPS imaging phenotypes (green-purple)
- **Lower triangle**: CROP-Seq sequencing phenotypes (blue-red)
- Ordered by joint hierarchical clustering

## Imports

In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from scipy.stats import zscore

plt.rcParams["svg.fonttype"] = "none"

## File paths

Explicit path to every input file used below.

**Before public release**, replace this cell with download instructions (or a pointer to the public dataset) and update the constants to match the released layout.

In [ ]:
# OPS gene-level embeddings (cell_dino, Phase-only)
OPS_PATH = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/phase_only/fixed_80%/cosine/gene_pca_optimized.h5ad"

# CROP-Seq sVAEplus perturbation-level embeddings (W matrix in .uns)
RNA_PATH = "/hpc/projects/data.science/duo.peng/sVAEplus/sVAEplus/6000HVG/svaeplus_results_2_256_1_200_0.5/svaeplus_embeddings.h5ad"

FIGURES_DIR = Path("../../output/figure_5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load and align OPS / RNA embeddings

Read OPS gene-level embeddings (filter NTCs) and the sVAE perturbation-level W matrix, then align both to the set of perturbations shared between the two modalities.

In [ ]:
# OPS embeddings (gene-level, one row per perturbation)
adata_ops    = ad.read_h5ad(OPS_PATH)
X_ops_all    = np.asarray(adata_ops.X, dtype=np.float32)
ops_names    = adata_ops.obs["perturbation"].values
ops_ntc_mask = np.array([n.startswith("NTC") for n in ops_names])
X_ops_nonntc = X_ops_all[~ops_ntc_mask]
ops_genes    = ops_names[~ops_ntc_mask]
print(f"OPS: {X_ops_all.shape[0]} total ({ops_ntc_mask.sum()} NTCs) → {X_ops_nonntc.shape[0]} perturbations × {X_ops_nonntc.shape[1]} features")

# sVAE embeddings (perturbation-level W matrix from .uns)
adata_rna    = ad.read_h5ad(RNA_PATH)
W_df         = adata_rna.uns["emb_pert_W_matrix"]
rna_genes    = np.array([g for g in W_df.index if g != "control"])
X_rna_nonntc = W_df.loc[rna_genes].values.astype(np.float32)
print(f"RNA: {X_rna_nonntc.shape[0]} perturbations × {X_rna_nonntc.shape[1]} features (sVAE W matrix)")

# Intersect shared perturbations and align both matrices to a common gene order
shared    = sorted(set(ops_genes) & set(rna_genes))
ops_only  = sorted(set(ops_genes) - set(rna_genes))
rna_only  = sorted(set(rna_genes) - set(ops_genes))
print(f"Shared: {len(shared)}  |  OPS-only: {len(ops_only)}  |  RNA-only: {len(rna_only)}")

ops_idx    = [np.where(ops_genes == g)[0][0] for g in shared]
rna_idx    = [np.where(rna_genes == g)[0][0] for g in shared]
X_ops      = X_ops_nonntc[ops_idx]
X_rna      = X_rna_nonntc[rna_idx]
gene_names = np.array(shared)
print(f"Aligned: {X_ops.shape[0]} perturbations  (OPS: {X_ops.shape[1]}d, RNA: {X_rna.shape[1]}d)")

## Centering, correlation, and joint clustering

Subtract the mean perturbation vector from each modality to remove the shared component, compute pairwise Pearson correlations within each modality, then run hierarchical clustering on a joint z-scored concatenation of the centred embeddings to produce a shared row/column ordering.

In [ ]:
# Center and compute within-modality correlation matrices
X_ops_c  = X_ops - X_ops.mean(axis=0, keepdims=True)
X_rna_c  = X_rna - X_rna.mean(axis=0, keepdims=True)
corr_ops = np.corrcoef(X_ops_c)
corr_rna = np.corrcoef(X_rna_c)
print(f"OPS corr range: [{corr_ops.min():.3f}, {corr_ops.max():.3f}]")
print(f"RNA corr range: [{corr_rna.min():.3f}, {corr_rna.max():.3f}]")

# Joint hierarchical clustering on concatenated z-scored centred embeddings
X_joint    = np.hstack([zscore(X_ops_c, axis=1), zscore(X_rna_c, axis=1)])
corr_joint = np.corrcoef(X_joint)
dist_joint = 1.0 - corr_joint
np.fill_diagonal(dist_joint, 0.0)
dist_joint = np.clip(dist_joint, 0, None)  # guard against float rounding

Z     = linkage(squareform(dist_joint, checks=False), method="average")
order = leaves_list(Z)
print(f"Joint clustering on {X_joint.shape[1]}-dim concatenated embedding")

## Save correlation matrices

Write the OPS and RNA correlation matrices to CSV (indexed by gene name) for downstream re-use.

In [ ]:
pd.DataFrame(corr_ops, index=gene_names, columns=gene_names).to_csv(FIGURES_DIR / "corr_ops.csv")
pd.DataFrame(corr_rna, index=gene_names, columns=gene_names).to_csv(FIGURES_DIR / "corr_rna.csv")
print(f"Saved OPS and RNA correlation matrices to {FIGURES_DIR}")

## Joint heatmap — paper figure

Split-triangle Pearson correlation heatmap with rows/columns ordered by the joint clustering above. Saves an SVG for the paper.

In [ ]:
n = len(gene_names)
corr_ops_ord   = corr_ops[np.ix_(order, order)]
corr_rna_ord   = corr_rna[np.ix_(order, order)]
gene_names_ord = gene_names[order]

# Mask: upper triangle for OPS (imaging), lower triangle for RNA (sequencing)
upper = np.triu(np.ones((n, n), dtype=bool), k=1)
lower = np.tril(np.ones((n, n), dtype=bool), k=-1)

ops_data = np.where(upper, corr_ops_ord, np.nan)
rna_data = np.where(lower, corr_rna_ord, np.nan)

vlim = 1
fig, ax = plt.subplots(figsize=(18, 16))

# Lower triangle: RNA (sequencing) — blue-red
im_rna = ax.imshow(rna_data, cmap="RdBu_r", vmin=-vlim, vmax=vlim,
                   aspect="auto", interpolation="nearest")
# Upper triangle: OPS (imaging) — green-purple
im_ops = ax.imshow(ops_data, cmap="PRGn", vmin=-vlim, vmax=vlim,
                   aspect="auto", interpolation="nearest")
# Diagonal
diag_data = np.full((n, n), np.nan)
np.fill_diagonal(diag_data, 1.0)
ax.imshow(diag_data, cmap=mcolors.ListedColormap(["#2d2d2d"]),
          vmin=0, vmax=1, aspect="auto", interpolation="nearest")

# Gene labels
ax.set_xticks(range(n))
ax.set_xticklabels(gene_names_ord, rotation=90, fontsize=3)
ax.set_yticks(range(n))
ax.set_yticklabels(gene_names_ord, fontsize=3)

ax.set_xlabel("Perturbations ordered by joint phenotype clustering", fontsize=12)
ax.set_ylabel("Perturbations ordered by joint phenotype clustering", fontsize=12)
ax.set_title("Perturbations ordered by joint phenotype clustering", fontsize=14, pad=15)

# Colorbars
cb_ops = fig.colorbar(im_ops, ax=ax, fraction=0.02, pad=0.12, location="right")
cb_ops.set_label("Above diagonal: imaging phenotypes\n(Pearson Correlation)", fontsize=9)
cb_rna = fig.colorbar(im_rna, ax=ax, fraction=0.02, pad=0.06, location="bottom")
cb_rna.set_label("Below diagonal: sequencing phenotypes\n(Pearson Correlation)", fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "joint_heatmap.svg", dpi=300, bbox_inches="tight")
print(f"Saved to {FIGURES_DIR / 'joint_heatmap.svg'}")
plt.show()